<!-- torchleet:colab -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Fengyee/TorchLeet/blob/main/torch/hard/GAN/GAN.ipynb)

Check your work: `pip install -e python/` once from the repo root (on Colab: `!pip install -q "git+https://github.com/Fengyee/TorchLeet#subdirectory=python"`), then `from torchleet import check; check("gan", ...)`


## Problem: Write a GAN

### Problem Statement
Implement a **Generative Adversarial Network (GAN)** by completing the required sections. The GAN consists of a **Generator** that creates fake data and a **Discriminator** that classifies data as real or fake.

### Requirements
1. **Define the Generator Class**:
   - **Purpose**: Generate fake data that mimics the real data distribution.
   - **Layers**:
     - Start with a fully connected layer to map the latent space (random noise) to a higher-dimensional space.
     - Use activation functions like `ReLU` to introduce non-linearity.
     - Add additional layers to process the data and refine its structure.
     - The final layer should output data in the target shape. Use an activation function like `Tanh` for scaling.
   - **Forward Pass**: Implement the forward method to pass the input through the defined layers.

2. **Define the Discriminator Class**:
   - **Purpose**: Classify data as real or fake.
   - **Layers**:
     - Use fully connected layers to process the input and extract features.
     - Apply activation functions like `LeakyReLU` to prevent dead neurons and stabilize training.
     - The final layer should output a single probability (real or fake) using a `Sigmoid` activation.
   - **Forward Pass**: Implement the forward method to process the input through the layers.

3. **Train the GAN**:
   - Alternate between training the Generator and Discriminator.
   - Use binary cross-entropy loss for both models.
   - Monitor the loss and generated samples during training.

### The objective, and the details that matter
GANs are defined by the minimax game
$$
\min_G \max_D \; \mathbb{E}_{x \sim p_\text{data}}[\log D(x)] + \mathbb{E}_{z}[\log(1 - D(G(z)))]
$$

- **Discriminator step:** maximize the objective = minimize BCE with label 1 for real samples and label 0 for fakes. Generate the fakes, then **`.detach()`** them: this step must update only D, and detaching keeps the backward pass out of G (no wasted compute, no stray gradients left in G's `.grad`).
- **Generator step:** minimizing $\log(1 - D(G(z)))$ as written *saturates* — early in training D rejects fakes confidently, $D(G(z)) \approx 0$, and the gradient vanishes. In practice — as the original GAN paper already suggests — G uses the **non-saturating** loss instead: maximize $\log D(G(z))$, i.e. BCE of $D(G(z))$ against label **1**. Here the fake batch must **not** be detached — the gradient has to flow back through D into G.
- Call `zero_grad()` on the optimizer you are about to step, *before* its `backward()`. The G step also deposits gradients in D's parameters; they are harmless only because the next D step zeroes them first.
- At the equilibrium $D(x) = \tfrac12$ everywhere, so the D loss (real + fake) tends to $2\ln 2 \approx 1.386$ and the G loss to $\ln 2 \approx 0.693$ — not to zero.
- `Sigmoid` + `BCELoss` is used here for clarity; production code outputs a logit and uses `BCEWithLogitsLoss`, which is numerically stable.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

In [ ]:
# Define the Generator
#TODO: Implement the Generator class
class Generator(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(Generator, self).__init__()
        # Add layers for the generator

    def forward(self, x):
        # Define the forward pass for the generator
        ...

# Define the Discriminator
#TODO: Implement the Discriminator class
class Discriminator(nn.Module):
    def __init__(self, input_dim):
        super(Discriminator, self).__init__()
        # Add layers for the discriminator

    def forward(self, x):
        # Define the forward pass for the discriminator
        ...

In [ ]:
# Generate synthetic data for training
torch.manual_seed(42)
real_data = torch.rand(100, 1) * 2 - 1  # 100 samples in the range [-1, 1]

# Initialize models, loss, and optimizers
latent_dim = 10
data_dim = 1
G = Generator(latent_dim, data_dim)
D = Discriminator(data_dim)

criterion = nn.BCELoss()
optimizer_G = optim.Adam(G.parameters(), lr=0.001)
optimizer_D = optim.Adam(D.parameters(), lr=0.001)

In [ ]:
# Training loop
epochs = 1000
for epoch in range(epochs):
    real_labels = torch.ones(real_data.size(0), 1)
    fake_labels = torch.zeros(real_data.size(0), 1)

    # ---- Train Discriminator: real -> 1, fake -> 0 ----
    # TODO: sample latent noise, generate a fake batch (detached: this step updates only D),
    #       compute loss_D = BCE(D(real), 1) + BCE(D(fake), 0), then zero_grad / backward / step
    loss_D = ...

    # ---- Train Generator (non-saturating loss): make D call a fresh fake batch real ----
    # TODO: sample new noise, fake = G(z) WITHOUT detach, loss_G = BCE(D(fake), 1),
    #       then zero_grad / backward / step on the generator's optimizer
    loss_G = ...

    # Log progress every 100 epochs
    if (epoch + 1) % 100 == 0:
        print(f"Epoch [{epoch + 1}/{epochs}] - Loss D: {loss_D.item():.4f}, Loss G: {loss_G.item():.4f}")

In [ ]:
# Generate new samples with the trained Generator
latent_samples = torch.randn(5, latent_dim)
with torch.no_grad():
    generated_data = G(latent_samples)
    print(f"Generated data: {generated_data.tolist()}")

    # The real data is Uniform(-1, 1): mean 0, std 1/sqrt(3) = 0.577
    many = G(torch.randn(5000, latent_dim))
    print(f"Generated mean/std: {many.mean().item():.3f} / {many.std().item():.3f} "
          f"(real data: {real_data.mean().item():.3f} / {real_data.std().item():.3f})")

In [ ]:
# Auto-grade (one-time setup from the repo root: pip install -e python/)
from torchleet import check
check("gan", Generator=Generator, Discriminator=Discriminator)